In [9]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_web_traffic
from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()



Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [10]:
from src.data import load_sales, load_web_traffic, load_promotions, load_order_items, load_orders
from src.features import add_time_features, add_lag_features
import pandas as pd

DATA_ROOT = project_root / "data/datathon-2026-round-1"

In [11]:
df_order_items = load_order_items(data_root=DATA_ROOT)
df_order = load_orders(data_root=DATA_ROOT)

# Attach order date to each order-item row, then count promo_id_2 usage per day.
order_items_with_date = df_order_items.merge(
    df_order[["order_id", "date"]], on="order_id", how="left"
)

promo2_by_date = (
    order_items_with_date.assign(
        promo2_used=lambda x: x["promo_id_2"].notna() & (x["promo_id_2"] != 0)
    )
    .groupby("date", as_index=False)["promo2_used"]
    .sum()
    .rename(columns={"promo2_used": "promo2_total_used"})
)

In [24]:
promo2_by_date.head()

,date,promo2_total_used
0,2012-07-04,0
1,2012-07-05,0
2,2012-07-06,0
3,2012-07-07,0
4,2012-07-08,0


In [28]:
from src.pipelines import train_validate_models, fit_models_full
from src.models import SklearnRegressorWrapper, SklearnRegressorConfig
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)


df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])

df = pd.merge(df_sales, df_traffic, on="date", how="left")
# df = pd.merge(df, promo2_by_date, on="date", how="left")

df = add_time_features(df, date_col="date")

# returning_users = max(0, sessions - unique_visitors)
# df["returning_users"] = df["sessions"] - df["unique_visitors"]
#df["returning_users"] = df["returning_users"].clip(lower=0)

# returning_rate = (sessions - unique_visitors) / sessions
df["returning_rate"] = (df["sessions"] - df["unique_visitors"]) / df["sessions"]

# df = add_lag_features(df, target_col="returning_rate", lags=[3], date_col="date")

In [29]:
df1 = df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'])
df1["ratio"] = df1["Revenue"] / df1["COGS"]
targets = ["Revenue", "ratio"]
feature_cols = [col for col in df1.columns if col not in ["date", "Revenue", "COGS", *targets]]
print(feature_cols)

model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df1,
    features=feature_cols,
    targets=targets,
    model_config=model_config,
)
results["metrics"]



['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend', 'returning_rate']


{'Revenue': {'mae': 573786.5084695623,
  'rmse': 807047.3750420011,
  'mape': 22.178775258815975,
  'smape': 21.252031864993732,
  'r2': 0.7653736983429531},
 'ratio': {'mae': 0.06490293242251076,
  'rmse': 0.12318763703115902,
  'mape': 7.015191675792405,
  'smape': 5.993640234636333,
  'r2': 0.14706009013212917}}

In [23]:
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)

df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv")
X_traffic_test = df_traffic_test[["date", "sessions", "unique_visitors"]].drop(columns=["date"])
df = pd.merge(df_sales, df_traffic, on="date", how="left")
df = add_time_features(df, date_col="date")
df.drop(columns=['sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source'], inplace=True)
df["ratio"] = df["Revenue"] / df["COGS"].clip(lower=1e-6)
df["revenue"] = df["Revenue"]
targets = ["revenue", "ratio"]
feature_cols = [col for col in df.columns if col not in ["date", "Revenue", "COGS", *targets]]
print(feature_cols)
model_config = SklearnRegressorConfig(
    model_type="xgboost", 
)
results = train_validate_models(
    df=df,
    features=feature_cols,
    targets=targets,
    model_config=model_config,
)
results["metrics"]


['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_month_end', 'is_month_start', 'is_weekend']


{'revenue': {'mae': 585314.4008703831,
  'rmse': 812629.873432484,
  'mape': 22.379391480814636,
  'smape': 21.461053364244695,
  'r2': 0.7621165636029108},
 'ratio': {'mae': 0.06554961298502573,
  'rmse': 0.12459877276687019,
  'mape': 7.0762277630970525,
  'smape': 6.035691033310183,
  'r2': 0.127407017472694}}